# HLS NDVI DataCube (earthaccess + odc-stac) on Coiled

End-to-end: authenticate with Earthdata, search CMR for HLS, translate the
results to STAC with `to_stac()`, then lazily load the bands into an
`xarray` cube with `odc.stac` on a **Coiled** Dask cluster.

Notes on the pieces that matter:
- CMR short names are `HLSL30` (Landsat 8/9 C2 L2) and `HLSS30`
  (Sentinel-2 L2A). There is **no** public HLS STAC endpoint — we search
  via earthaccess and hand the STAC-ified items to odc-stac.
- The `to_stac()` items carry **HTTPS (indirect)** links, so GDAL reads
  them with the Earthdata **Bearer token** — we forward that env to the
  Coiled workers via `configure_rio(client=...)`.
- We do **not** pass `crs`/`resolution`: odc-stac auto-derives the output
  geobox (projection + resolution) from the items themselves.

## 0. Parameters

Edit these to change the study area, time window, sensor, and bands.

In [ ]:
# ---- Study area ----
# bbox is (west, south, east, north) in WGS84 lon/lat
bbox = (-109.05, 37.0, -101.91, 41.0)  # Colorado

# ---- Time window ----
temporal = ("2023-01-01", "2023-06-30")

# ---- Sensor (CMR short name) ----
# "HLSL30" = Landsat 8/9 C2 L2 (30 m)   red/nir = B04 / B05
# "HLSS30" = Sentinel-2 L2A   (30 m)    red/nir = B04 / B08
# Use a single sensor for one clean, comparable NDVI series.
short_name = "HLSL30"

# ---- Bands to load (asset keys) ----
bands = ["B04", "B05"]  # Landsat C2: B04=red, B05=NIR

# ---- NDVI band pair: (nir, red) ----
NIR, RED = "B05", "B04"

# ---- Load tuning ----
GROUPBY = "solar_day"  # collapse overlapping scenes to ~1/day
DTYPE = "uint16"
NODATA = 0

## 1. Instantiate the Coiled Dask cluster

Run this cell first. `coiled.Cluster()` returns a `dask.distributed.Client`
wired to your Coiled account (uses `COILED_API_KEY` / the `coiled` config).

In [ ]:
import coiled

client = coiled.Cluster()  # -> dask.distributed.Client on your Coiled cluster
client

## 2. Authenticate and search CMR for HLS

In [ ]:
import earthaccess

auth = earthaccess.login()  # NASA Earthdata Login
auth

In [ ]:
granules = earthaccess.search_data(
    short_name=short_name,
    temporal=temporal,
    bounding_box=bbox,
)
granules

## 3. Translate the CMR results into STAC items

In [ ]:
items = granules.to_stac()
print(f"Found {len(items)} STAC items")

## 4. Forward the Earthdata auth env to the Coiled workers

Because the `to_stac()` links are HTTPS, GDAL needs the Bearer token. We pass
the same env to `configure_rio(client=...)` so **every worker** inherits it
(a local `rasterio.Env` alone would not reach the workers, and reads would 401).

In [ ]:
from odc.stac import configure_rio

configure_rio(
    client=client,
    GDAL_HTTP_HEADERS=f"Authorization: Bearer {auth.token['access_token']}",
    CPL_VSIL_CURL_USE_HEAD="NO",
    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
    VSI_CACHE=True,
)

## 5. Build the cube (projection auto-wired)

No `crs`/`resolution` passed — odc-stac derives the output geobox (projection
+ resolution) from the items' own metadata. We clip to `bbox` and group by
solar day.

In [ ]:
from odc.stac import stac_load

ds = stac_load(
    items,
    bands=bands,
    bbox=bbox,
    groupby=GROUPBY,
    dtype=DTYPE,
    nodata=NODATA,
)
ds

## 6. Compute NDVI and plot

In [ ]:
ds["NDVI"] = (ds[NIR] - ds[RED]) / (ds[NIR] + ds[RED])

ndvi = ds["NDVI"].compute()
ndvi.plot(col="time", col_wrap=6, vmin=0, vmax=1, aspect="auto")

In [ ]:
client.close()